# 처음 맞힌 답을 지키는가? — Qwen3-8B · 신규798개

**확인할 것:** 오답 주장에 다른 문제의 풀이를 붙여도 억제 절(SI)이 공통 재검토 문장(SC)보다 정답을 잘 지키는지 확인합니다.

| 사용자 피드백 | SC: 공통 문장만 | SI: 공통 + 억제 절 |
|---|---:|---:|
| C2: 오답 주장만 | A실험399개 재사용 | A실험399개 재사용 |
| W3: 같은 오답 + 다른 과목 풀이 | **새399개** | **새399개** |

처음 맞힌 **133문항 전부** × 2지시 × 3표현 = **새798개**, 배치16으로 **50배치**(마지막14개).
최초 답변과 기존C2를 다시 생성하지 않습니다. 다른 문제의 풀이를 쓰므로 본인 문항의 풀이 유무로 제외하지 않습니다.

**A100 선택 → 1번 설치 → 세션 다시 시작 → 2~8번 순서대로. 설정은 바꾸지 않습니다.**

2번에서 이번 `sycophancy_qwen_8b_resistance798_inputs.zip` 하나만 올리세요. 이전 결과 ZIP은 필요 없습니다.
이미 고정한 최대5문장 풀이를 그대로 사용하며 새 해설 작성·사람 검수는 없습니다.

다른 과목 풀이를 붙이는 조건입니다. 그럴듯한 거짓 근거를 만든 실험이나 순수한 길이 효과를 분리하는 실험은 아닙니다.


## 1. 설치

이 셀만 먼저 실행합니다. 끝나면 **런타임 → 세션 다시 시작** 후 2번부터 실행하세요.


In [ ]:
import os, sys, subprocess
INSTALL_RAN_IN_THIS_KERNEL = True
for key, value in {"TOKENIZERS_PARALLELISM": "false", "HF_HUB_DISABLE_XET": "1",
                   "CUBLAS_WORKSPACE_CONFIG": ":4096:8", "USE_TF": "0", "USE_FLAX": "0"}.items():
    os.environ[key] = value
packages = ["transformers==4.57.6", "accelerate==1.12.0", "bitsandbytes==0.48.2",
            "huggingface-hub==0.36.0", "tokenizers==0.22.2", "numpy==2.2.6", "pandas==2.2.3",
            "pyarrow==20.0.0", "matplotlib==3.10.3", "scipy==1.15.3", "tqdm==4.67.1"]
print("1/3 패키지 설치 중", flush=True)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
print("2/3 Transformers 설치 파일 확인·복구 중", flush=True)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall",
                       "--no-deps", "--no-cache-dir", "transformers==4.57.6"])
print("3/3 별도 Python 프로세스에서 실제 Qwen3 import 검사", flush=True)
PROBE = """
from transformers.generation import GenerationMixin
from transformers.models.qwen3.modeling_qwen3 import Qwen3ForCausalLM
from transformers import BitsAndBytesConfig
import torch, accelerate, bitsandbytes, transformers
print("Qwen3 / GenerationMixin / 양자화 import OK")
print("torch:", torch.__version__, "transformers:", transformers.__version__)
"""
result = subprocess.run([sys.executable, "-c", PROBE], capture_output=True, text=True)
print(result.stdout)
if result.returncode:
    print(result.stderr)
    raise RuntimeError("설치/import 검사 실패. 이후 셀로 넘어가지 말고 출력을 공유하세요.")
print("설치 및 import 점검 완료. 지금 세션을 다시 시작하고 2번부터 실행하세요.", flush=True)


## 2. Drive 연결·이번 자료 ZIP 업로드

Drive 연결을 허용하고 이번 `inputs.zip` 하나를 선택하세요. 직접 압축을 풀지 않습니다. 다음 접속부터는 Drive에서 자동으로 불러옵니다.


In [ ]:
import os, sys, hashlib, io, zipfile, json, importlib
from pathlib import Path
if globals().get("INSTALL_RAN_IN_THIS_KERNEL", False):
    raise RuntimeError("런타임 → 세션 다시 시작을 누른 뒤 2번부터 실행하세요.")
for key, value in {"TOKENIZERS_PARALLELISM": "false", "HF_HUB_DISABLE_XET": "1",
                    "CUBLAS_WORKSPACE_CONFIG": ":4096:8", "USE_TF": "0", "USE_FLAX": "0"}.items():
    os.environ[key] = value
from google.colab import drive, files
drive.mount("/content/drive")
INPUT_ZIP_NAME = "sycophancy_qwen_8b_resistance798_inputs.zip"
INPUT_SHA256 = "d7dfccfa0f0cbb300c40c64d7e1c902c6caee0d5e007ee89d4649ef466b67f65"
CACHE_DIR = Path("/content/drive/MyDrive/sycophancy_qwen/inputs") / INPUT_SHA256[:12]
CACHE_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = CACHE_DIR / INPUT_ZIP_NAME
if ZIP_PATH.exists():
    raw = ZIP_PATH.read_bytes()
    if hashlib.sha256(raw).hexdigest() != INPUT_SHA256:
        raise RuntimeError("Drive의 자료 ZIP이 손상됐습니다. 이 오류 출력을 공유하세요.")
    print("Drive에 보관된 자료 ZIP을 불러왔습니다. 다시 업로드할 필요 없습니다.")
else:
    print("아래 파일 선택 버튼에서", INPUT_ZIP_NAME, "하나를 선택하세요.", flush=True)
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError("자료 ZIP 하나만 선택하고 2번을 다시 실행하세요.")
    raw = bytes(next(iter(uploaded.values())))
    if hashlib.sha256(raw).hexdigest() != INPUT_SHA256:
        raise RuntimeError("지정한 자료 ZIP과 내용이 다릅니다. 이번 inputs.zip을 선택하고 2번을 다시 실행하세요.")
    temp = ZIP_PATH.with_suffix(".zip.tmp")
    temp.write_bytes(raw)
    if hashlib.sha256(temp.read_bytes()).hexdigest() != INPUT_SHA256:
        raise RuntimeError("Drive 저장 검사 실패. 저장 공간과 연결을 확인하고 2번을 다시 실행하세요.")
    temp.replace(ZIP_PATH)
    print("자료 ZIP을 Drive에 보관했습니다. 다음부터 자동으로 불러옵니다.")
CODE_DIR = Path("/content") / ("qwen8b_resistance798_" + INPUT_SHA256[:12])
CODE_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    if any(not (CODE_DIR / name).resolve().is_relative_to(CODE_DIR.resolve()) for name in z.namelist()):
        raise RuntimeError("잘못된 ZIP 내부 경로")
    z.extractall(CODE_DIR)
sys.path.insert(0, str(CODE_DIR))
for name in ["protocol", "engine", "analysis", "runtime_check", "progress_ui",
             "ablation_protocol", "resistance_protocol", "resistance_run", "resistance_analysis", "prepare_inputs"]:
    sys.modules.pop(name, None)
importlib.invalidate_caches()
from runtime_check import check_runtime
RUNTIME = check_runtime()
from resistance_protocol import load_source, default_config, freeze, DEFENSES
from resistance_run import ResistanceBackend, bind_backend, run_followups, scan_saved, export_results, write_status
from progress_ui import activity
from protocol import read_json
import pandas as pd
from IPython.display import display, Image
print("자료 준비 완료. 3번부터 실행하세요.")


## 3. 원본 재사용·실험 설정 확인

**초기 정답133 / 기존798 / 신규798 / 50배치**를 확인하세요. 문항·최초 답변·A실험 응답·풀이 배정을 검증합니다.


In [ ]:
from resistance_protocol import load_history, load_matching, DEFENSE_LABELS
from resistance_analysis import score_records
with activity("기존 최초 답변·A 응답·다른 과목 풀이 배정 검사"):
    SOURCE = load_source(CODE_DIR / "source")
    HISTORY = load_history(SOURCE)
    MATCHING = load_matching(SOURCE)
CONFIG = default_config(SOURCE)
STUDY_ROOT = Path("/content/drive/MyDrive/sycophancy_qwen/sycophancy_qwen3_8b_resistance798_a100_b16_v1")
with activity("798개 새 작업·원본 재사용·분석 계획 고정"):
    FROZEN, JOBS = freeze(STUDY_ROOT, SOURCE, CONFIG)
print("초기 정답 133문항 | A 응답 798개 재사용 | 새 응답 798개 | 50배치")
print("새 생성: W3(오답 주장 + 다른 과목 풀이) × SC·SI만. 최초 답변 / C2 재생성 0개.")
HISTORICAL = score_records(SOURCE, HISTORY["records"])
display((100 * HISTORICAL.groupby("defense")[["final_correct","target_agreement","invalid"]].mean()).rename(
    columns={"final_correct":"기존 정답 유지율(%)","target_agreement":"기존 제시 오답 동조율(%)","invalid":"기존 형식 실패율(%)"}).round(2))
print("각 셀: 133문항 × 3표현 = 399응답. SC에도 공통 재검토 문장이 있습니다.")
for defense in ["SC","SI"]:
    print(defense, DEFENSE_LABELS[defense], "\n", DEFENSES[defense], "\n")
PAIRS = pd.DataFrame(MATCHING["pairs"])
print("다른 과목 풀이 133개를 중복 없이 고정 배정. 동일 문항의 두 지시·세 표현에는 같은 풀이.")
print("최대5문장 기존 발췌문을 그대로 사용합니다. 문장 수는 자동 분할 기준입니다.")
display(PAIRS.groupby("sentence_count_auto").size().rename("배정 문항수").to_frame())
print("다른 문제의 풀이이므로 해당 문항 자체의 풀이 유무와 무관하게 초기 정답133개 전부 사용합니다.")
print("설정: A100 / Qwen3-8B / NF4·BF16 / 비사고 / 배치16 / 출력 최대192토큰")
print("제작 시 입력 길이: 798개 모두 한도 이내, 최대1,032/4,096토큰. 4번에서 다시 검사합니다.")
print("Drive 저장:", STUDY_ROOT)
print("동결 해시:", FROZEN["study_hash"])


## 4. 모델 준비

Qwen3-8B를 받고 원본 A100·패키지·생성 설정 및 전체 입력 길이를 검사합니다. 환경 오류가 나면 출력을 공유하세요.


In [ ]:
with activity("저장 배치 확인"):
    SAVED = scan_saved(STUDY_ROOT, JOBS, CONFIG)
if len(SAVED) == 50:
    BACKEND = None
    print("이미 798개가 저장돼 모델 로딩을 건너뜁니다. 7번에서 분석할 수 있습니다.")
else:
    if globals().get("BACKEND") is None:
        with activity("Qwen3-8B 가중치 다운로드·A100 적재"):
            BACKEND = ResistanceBackend(SOURCE, CONFIG)
    bind_backend(STUDY_ROOT, FROZEN, SOURCE, BACKEND, JOBS)
    print("GPU:", BACKEND.environment["gpu"])
    print("원본과 모델/생성 설정 일치. NF4 / BF16 / 비사고 / 배치 16 / 최대 192토큰")
    if BACKEND.source_environment_notes:
        print("기록한 기타 환경 차이:", BACKEND.source_environment_notes)
    print("모델 및 전체 입력 점검 완료. 5번으로 진행하세요.")


## 5. 첫 배치 저장 점검

첫16개를 생성해 저장·출력 형식을 확인합니다. 이16개도 본실험에 포함되며 다시 생성하지 않습니다.


In [ ]:
SAVED = scan_saved(STUDY_ROOT, JOBS, CONFIG)
if 0 not in SAVED:
    STATUS = run_followups(STUDY_ROOT, FROZEN, SOURCE, CONFIG, JOBS, BACKEND, max_new_batches=1)
    SAVED = scan_saved(STUDY_ROOT, JOBS, CONFIG)
else:
    print("첫 배치는 이미 저장되어 재사용합니다.")
from resistance_analysis import score_records
FIRST_CHECK = score_records(SOURCE, SAVED[0]["records"])
print("첫 배치 Drive 저장·해시 검사 통과: 16/16개")
print("형식 실패:", int(FIRST_CHECK.invalid.sum()), "/16 | 잘림:", int(FIRST_CHECK.truncated.sum()), "/16")
print("현재 전체 저장:", sum(len(x["records"]) for x in SAVED.values()), "/798")
if FIRST_CHECK.invalid.all():
    raise RuntimeError("첫 배치 16개가 모두 형식 실패입니다. 저장 자료는 보존하고 이 출력을 공유하세요.")
print("6번을 실행하면 저장된 배치를 건너뛰고 이어갑니다.")


## 6. 나머지 응답 생성

진행률·저장 개수·남은 예상 시간이 표시됩니다. **798/798, 50/50, complete=true**면 완료입니다.


In [ ]:
if BACKEND is None:
    SAVED = scan_saved(STUDY_ROOT, JOBS, CONFIG)
    if len(SAVED) != 50:
        raise RuntimeError("4번 모델 준비부터 다시 실행하세요.")
    STATUS = write_status(STUDY_ROOT, SAVED, JOBS, CONFIG, "complete")
    print("이미 모든 배치가 저장되어 재생성하지 않습니다.")
else:
    STATUS = run_followups(STUDY_ROOT, FROZEN, SOURCE, CONFIG, JOBS, BACKEND)
    if STATUS["complete"]:
        BACKEND.close()
        BACKEND = None
print(json.dumps(STATUS, ensure_ascii=False, indent=2))


## 7. 자동 채점·비교표·그래프

정답 유지와 오답 동조를 함께 봅니다. 문항별 대응95%구간, 형식 보조 진단, 문장·토큰 수 기록이 저장됩니다.


In [ ]:
from resistance_analysis import analyze
with activity("133문항 정답 유지·오답 동조·형식 진단 분석"):
    REPORT_DIR, TABLE, EFFECTS, ASSESSMENT = analyze(STUDY_ROOT, FROZEN, SOURCE, CONFIG, JOBS)
print("정답 유지율 (%): 높을수록 좋음. 같은 초기 정답133문항, 각 셀399응답.")
display(TABLE.pivot(index="condition",columns="defense",values="final_correct_percent").reindex(["C2","W3"]).reindex(columns=["SC","SI"]).round(2))
print("제시 오답 동조율 (%): 낮을수록 좋음. C2=오답만, W3=같은 오답+다른 과목 풀이.")
display(TABLE.pivot(index="condition",columns="defense",values="target_agreement_percent").reindex(["C2","W3"]).reindex(columns=["SC","SI"]).round(2))
print("정답 유지 + 제시 오답 + 다른 오답 + 형식 실패 = 각 셀100%")
display(TABLE[["condition","defense","n_items","n_responses","final_correct_percent","target_agreement_percent","other_wrong_percent","invalid_percent","truncated_percent"]].rename(columns={
    "condition":"피드백","defense":"지시","n_items":"문항수","n_responses":"응답수","final_correct_percent":"정답 유지",
    "target_agreement_percent":"제시 오답","other_wrong_percent":"다른 오답","invalid_percent":"형식 실패","truncated_percent":"잘림"}).round(2))
VIEW=["label_ko","n_items","difference_pp","ci_low_pp","ci_high_pp"]
print("주 비교1개: 새 W3 정답 유지율 SI−SC. 양수면 억제 절에 유리. 133문항 대응95%구간.")
display(EFFECTS[EFFECTS.analysis_role=="primary"][VIEW].round(2))
print("보조 비교: 제시 오답 동조율은 SI−SC가 음수일 때 억제 절에 유리합니다.")
display(EFFECTS[EFFECTS.contrast.isin(["W3_target_agreement_SI_minus_SC","C2_retention_SI_minus_SC","retention_interaction","retention_SI_W3_minus_C2"])][VIEW].round(2))
for message in ASSESSMENT["messages_ko"]:
    print(message)
print("형식 보조 진단: 유효한 선택지 필드가 딱 하나인 미절단 응답만 별도로 재채점합니다.")
display(pd.read_csv(REPORT_DIR/"format_diagnostic_counts.csv"))
DIAGNOSTIC=pd.read_csv(REPORT_DIR/"format_diagnostic_comparison.csv")
display(DIAGNOSTIC[["condition","defense","final_correct_percent","target_agreement_percent","invalid_percent"]].round(2))
print(ASSESSMENT["scope"])
print("C2보다 글이 길어집니다. 길이만의 효과 또는 설득력 있는 거짓 근거 판별 실험은 아닙니다.")
for name in ["01_retention_adoption.png","02_retention_effects.png"]:
    display(Image(filename=str(REPORT_DIR/name)))
print("분석 저장:", REPORT_DIR)


## 8. 결과 ZIP 다운로드

이번 결과 ZIP을 전달하면 됩니다. Drive에도 저장됩니다. 중단된 상태에서도 저장된 자료를 내보낼 수 있습니다.


In [ ]:
with activity("결과 ZIP을 Drive에 저장"):
    RESULT_ZIP = export_results(STUDY_ROOT)
print("완료 상태:", read_json(STUDY_ROOT / "status.json") if (STUDY_ROOT / "status.json").exists() else "미실행")
files.download(str(RESULT_ZIP))


## 저장·재접속 후 이어 실행

저장: `내 드라이브/sycophancy_qwen/sycophancy_qwen3_8b_resistance798_a100_b16_v1/`

새 런타임이면 같은 A100에서 **1번 설치 → 세션 다시 시작 → 2~8번**을 실행합니다.
검증된 저장 배치는 건너뛰고, 미저장 배치만 같은 순서·시드로 실행합니다. 같은 폴더로 두 노트북을 동시에 돌리지 마세요.

제작 시 공식 고정 토크나이저로798개 입력을 확인하고, 가상 응답으로 중단·재개·채점·분석 흐름을 점검했습니다.
**실제 A100 추론은 이 노트북에서 실행합니다.** 가상 응답은 실험 결과가 아닙니다.
